# Movie Review Classifier - Naive Bayes
In this notebook I build a 3-class text classifier using a Multinomial Naive Bayes model. 

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

## 1. Data Loading and Preprocessing

I load the training and test sets from CSV files using the same preprocessing steps as my other notebooks -- dropping rows with missing TEXT from the training set and filling missing values in the test set with empty strings.

In [2]:
class TextPreprocessor:
    """handles loading and cleaning the data"""
    
    def __init__(self, train_path, test_path):
        self.train_path = train_path
        self.test_path = test_path
    
    def load(self):
        train = pd.read_csv(self.train_path)
        test = pd.read_csv(self.test_path)
        
        train = train.dropna(subset=["TEXT"])  # drop rows with missing text
        test = test.fillna({"TEXT": ""})        # fill missing test text with empty string
        
        return train, test


# load data
preprocessor = TextPreprocessor("../train.csv", "../test.csv")
train, test = preprocessor.load()

X_train = train["TEXT"]
y_train = train["LABEL"]
X_test = test["TEXT"]

print("Train size:", len(train))
print("Test size:", len(test))
print("Class distribution:\n", y_train.value_counts())

Train size: 70310
Test size: 17580
Class distribution:
 LABEL
0    32282
1    19139
2    18889
Name: count, dtype: int64


## 2. Feature Extraction

I convert raw text into numerical features using TF-IDF with unigrams and bigrams. One thing to note is that Multinomial Naive Bayes requires non-negative input values, which TF-IDF satisfies since all values are between 0 and 1. Reference: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html.

In [3]:
class FeatureExtractor:
    """converts raw text into TF-IDF features"""
    
    def __init__(self, ngram_range=(1, 2), max_features=50000):
        self.vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    
    def fit(self, texts):
        self.vectorizer.fit(texts)
    
    def transform(self, texts):
        return self.vectorizer.transform(texts)
    
    def fit_transform(self, texts):
        return self.vectorizer.fit_transform(texts)

## 3. Classifier

I use Multinomial Naive Bayes as my classifier. Naive Bayes is a probabilistic classifier based on Bayes' theorem with a "naive" assumption that all features are independent of each other given the class label. Despite this oversimplification, it tends to work surprisingly well on text classification tasks and is very fast to train.

I chose `MultinomialNB` specifically because it's designed for discrete count-like features, making it a natural fit for TF-IDF vectors. One limitation is that unlike Logistic Regression, Naive Bayes doesn't have a `class_weight` parameter, so it may be more affected by the class imbalance in this dataset. I used https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html and in the https://scikit-learn.org/stable/modules/naive_bayes.html.

In [4]:
class MovieReviewClassifier:
    """wraps MultinomialNB for 3-class classification"""
    
    def __init__(self):
        self.clf = MultinomialNB()
    
    def train(self, features, labels):
        self.clf.fit(features, labels)
    
    def predict(self, features):
        return self.clf.predict(features)
    
    def evaluate(self, X, y, cv=5):
        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=50000)),
            ("clf", MultinomialNB())
        ])
        skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=66)
        scores = cross_val_score(pipeline, X, y, cv=skf, scoring="f1_macro")
        return scores

## 4. Cross Validation

I evaluate using 5-fold stratified cross validation and report the macro F1 score, consistent with the Kaggle evaluation metric for this competition.

In [5]:
classifier = MovieReviewClassifier()
scores = classifier.evaluate(X_train, y_train)

print("Naive Bayes CV Macro F1 scores:", scores)
print("Mean:", scores.mean().round(4))
print("Std:", scores.std().round(4))

Naive Bayes CV Macro F1 scores: [0.89540088 0.89406832 0.8912273  0.89696998 0.89567238]
Mean: 0.8947
Std: 0.002


### Cross Validation Results

I evaluated Naive Bayes using 5-fold stratified cross validation and got a mean macro F1 of **0.8948** with a standard deviation of **0.0019**. The low standard deviation tells me the model is consistent across folds, but the overall score is noticeably lower than both my Logistic Regression (0.9197) and SVM (0.9200) models. This gap makes sense when I think about the core assumption Naive Bayes makes -- it treats every feature as independent of every other feature given the class label. In reality, word combinations in reviews carry a lot of meaning (e.g. "not good" vs "good"), so that independence assumption hurts performance here.

I also noticed from the sklearn documentation that `MultinomialNB` doesn't support `class_weight`, unlike Logistic Regression and LinearSVC. This means the model has no built-in way to compensate for the class imbalance I found in EDA, where class 0 made up about 46% of the training data. That likely contributed to the lower F1 score as well. One thing I could try to address this is manually computing sample weights and passing them through the `fit` method, but given that LR already outperforms NB significantly I decided it wasn't worth pursuing further.

## 5. Training and Prediction

In [6]:
extractor = FeatureExtractor()
X_train_features = extractor.fit_transform(X_train)
X_test_features = extractor.transform(X_test)

classifier.train(X_train_features, y_train)
predictions = classifier.predict(X_test_features)

print("Prediction distribution:")
print(pd.Series(predictions).value_counts())

Prediction distribution:
0    7970
1    4859
2    4751
Name: count, dtype: int64


### Prediction Distribution

The model predicted 7,970 instances as class 0 (not a review), 4,859 as class 1 (positive), and 4,751 as class 2 (negative). The distribution is fairly proportional to what I saw in the training data during EDA, which is somewhat surprising given that Naive Bayes has no `class_weight` parameter to explicitly handle the imbalance. That said, class 0 is still the most predicted class by a decent margin, which aligns with it being the majority class in training. Comparing this to my Logistic Regression predictions (8,051 / 4,879 / 4,650), the distributions are very similar, though NB predicts slightly fewer class 0 and slightly more class 2 instances. Despite the similar prediction distributions, the lower macro F1 score tells me that NB is making more mistakes on the boundaries between classes -- particularly likely between class 0 and classes 1 and 2, since those are the trickiest cases I identified during EDA.

## 6. Generate Submission File

In [7]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "LABEL": predictions
})

submission.to_csv("../submissions/submission_nb.csv", index=False)
print(submission.head())

                     ID  LABEL
0   1087873697464833975      0
1   5853461517618045821      1
2   1225516091890726770      2
3  11795874926011119457      0
4  15956464546963841646      0
